In [ ]:
import pandas as pd
import numpy as np

# ===============================
# 1️⃣ Load data
# ===============================
path_w1 = r"C:\Users\INFOTEC\OneDrive\Bureau\Preprocessing_new_w1\corrected\Logistics_Preprocessed_Corrected_with_Score.xlsx"
path_w2 = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_new_w2\corrected\Logistics_Preprocessed_Corrected_with_Score.xlsx"

w1 = pd.read_excel(path_w1)
w2 = pd.read_excel(path_w2)

# ===============================
# 2️⃣ Base cleaning
# ===============================
def clean(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df.replace(["", " ", "NaN"], np.nan, inplace=True)
    return df

w1 = clean(w1)
w2 = clean(w2)

# ===============================
# 3️⃣ Merge by PartNumber
# ===============================
KEY = "PartNumber"

df = w1.merge(
    w2,
    on=KEY,
    how="outer",
    suffixes=("_W1", "_W2")
)

# ===============================
# 4️⃣ UNIT PRICE CORRECTION
# ===============================
df["Unit_Price"] = df["Unit_Price_W1"]

df["Unit_Price"] = np.where(
    df["Unit_Price"].isna(),
    df["Unit_Price_W2"],
    df["Unit_Price"]
)

median_price = df["Unit_Price"].median()
df["Unit_Price"].fillna(median_price, inplace=True)

# ===============================
# 5️⃣ INVENTORY CLEANING
# ===============================
for col in ["Inventory_W1", "Inventory_W2"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)
        df.loc[df[col] < 0, col] = 0

# ===============================
# 6️⃣ WEEKLY USAGE CALCULATION
# ===============================
df["Weekly_Usage"] = df["Weekly_Usage_W2"]

mask_missing_usage = df["Weekly_Usage"].isna()

df.loc[mask_missing_usage, "Weekly_Usage"] = (
    df.loc[mask_missing_usage, "Inventory_W1"]
    - df.loc[mask_missing_usage, "Inventory_W2"]
)

df.loc[df["Weekly_Usage"] < 0, "Weekly_Usage"] = 0

# ===============================
# 7️⃣ REAL INVENTORY W2
# ===============================
df["Real_Inventory_W2"] = (
    df["Inventory_W1"] - df["Weekly_Usage"]
)

df.loc[df["Real_Inventory_W2"] < 0, "Real_Inventory_W2"] = 0

# ===============================
# 8️⃣ FINAL DATASETS
# ===============================
w1_final = df[
    [KEY, "Inventory_W1", "Unit_Price"]
].rename(columns={
    "Inventory_W1": "Inventory"
})

w2_final = df[
    [KEY, "Real_Inventory_W2", "Weekly_Usage", "Unit_Price"]
].rename(columns={
    "Real_Inventory_W2": "Inventory"
})

# ===============================
# 9️⃣ Export Excel
# ===============================
output = r"C:\Users\INFOTEC\OneDrive\Bureau\Weekly_Logistics_Corrected_FINAL.xlsx"

with pd.ExcelWriter(output, engine="xlsxwriter") as writer:
    w1_final.to_excel(writer, sheet_name="Week1_Final", index=False)
    w2_final.to_excel(writer, sheet_name="Week2_Final", index=False)
    df.to_excel(writer, sheet_name="Merged_Debug", index=False)

print("✅ Correction métier + preprocessing terminé")
print("📁 File :", output)
